# Advanced Quant Research Tutorial: Multi-Asset Portfolio Strategies

**Learning objectives:**
- Load diverse asset classes: FX, Crypto, Bonds, Equities
- Implement multiple strategy types with Backtrader
- Build FX Carry trading strategy
- Create Crypto mean reversion strategy
- Combine all into multi-asset portfolio
- Perform walk-forward optimization

This tutorial demonstrates advanced multi-asset research with real-world complexity using Backtrader.


## 1. Setup

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path('/Users/zelin/Desktop/PA Investment/Invest_strategy')
sys.path.insert(0, str(project_root))

import backtrader as bt
from backend.backtest_engine import BacktestEngine, IBKRDataFeed

print("Setup complete")

## 2. Load Multiple Asset Classes

In [ ]:
import yfinance as yf

# Load diverse asset classes
asset_config = {
    'FX': {'ticker': 'EURUSD=X', 'name': 'EUR/USD'},
    'Crypto': {'ticker': 'BTC-USD', 'name': 'Bitcoin'},
    'Bonds': {'ticker': 'TLT', 'name': '20+ Year Treasury'},
    'Equities': {'ticker': 'IVV', 'name': 'S&P 500 ETF'}
}

start_date = "2020-01-01"
end_date = "2024-12-31"

data_dict = {}
for asset_type, config in asset_config.items():
    df = yf.download(config['ticker'], start=start_date, end=end_date, progress=False)
    if len(df) > 0:
        df = df.droplevel('Ticker', axis=1)
        df = df.reset_index()
        df.columns = ['date', 'open', 'high', 'low', 'close', 'volume']
        data_dict[asset_type] = df
        print(f"Loaded {len(df)} days of {config['name']} data")

print(f"\nTotal assets loaded: {len(data_dict)}")

## 3. Strategy 1: FX Carry / Momentum

In [ ]:
class FXMomentumStrategy(bt.Strategy):
    """
    FX momentum strategy.
    - Long EUR/USD when price above 50-day SMA
    - Short when below
    """
    params = (
        ('sma_period', 50),
    )
    
    def __init__(self):
        self.sma = bt.indicators.SimpleMovingAverage(
            self.data.close, period=self.params.sma_period
        )
        
    def next(self):
        if len(self) < self.params.sma_period:
            return
            
        if not self.position:
            if self.data.close[0] > self.sma[0]:
                self.buy()
        else:
            if self.data.close[0] < self.sma[0]:
                self.sell()

# Run FX strategy
engine_fx = BacktestEngine(cash=100000, commission=0.0001)  # Lower commission for FX
engine_fx.add_data(IBKRDataFeed(dataname=data_dict['FX'].copy()), name='EURUSD')
engine_fx.add_strategy(FXMomentumStrategy)
result_fx = engine_fx.run_backtest()

print("=== FX Momentum Strategy Results ===")
print(f"Final Value: ${result_fx['final_value']:,.0f}")
print(f"Total Return: {result_fx['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {result_fx['sharpe_ratio']:.3f}")

## 4. Strategy 2: Crypto Mean Reversion

In [ ]:
class CryptoMeanReversion(bt.Strategy):
    """
    Crypto mean reversion using Bollinger Bands.
    - Long when price touches lower band
    - Short when price touches upper band
    """
    params = (
        ('period', 20),
        ('devfactor', 2.0),
    )
    
    def __init__(self):
        self.bb = bt.indicators.BollingerBands(
            self.data.close, period=self.params.period, devfactor=self.params.devfactor
        )
        
    def next(self):
        if len(self) < self.params.period:
            return
            
        upper = self.bb.lines.top[0]
        lower = self.bb.lines.bot[0]
        close = self.data.close[0]
        
        if not self.position:
            if close < lower:
                self.buy()
        else:
            if close > upper:
                self.sell()
                
# Run Crypto strategy
engine_crypto = BacktestEngine(cash=100000, commission=0.001)
engine_crypto.add_data(IBKRDataFeed(dataname=data_dict['Crypto'].copy()), name='BTC')
engine_crypto.add_strategy(CryptoMeanReversion)
result_crypto = engine_crypto.run_backtest()

print("=== Crypto Mean Reversion Results ===")
print(f"Final Value: ${result_crypto['final_value']:,.0f}")
print(f"Total Return: {result_crypto['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {result_crypto['sharpe_ratio']:.3f}")

## 5. Strategy 3: Bond Volatility (Flight to Quality)

In [ ]:
class BondVolatilityStrategy(bt.Strategy):
    """
    Bond strategy using volatility regime.
    - Long bonds when equity volatility is high (flight to quality)
    """
    params = (
        ('sma_period', 50),
    )
    
    def __init__(self):
        # Simple version: trend following on bonds
        self.sma = bt.indicators.SimpleMovingAverage(
            self.data.close, period=self.params.sma_period
        )
        
    def next(self):
        if len(self) < self.params.sma_period:
            return
            
        if not self.position:
            if self.data.close[0] > self.sma[0]:
                self.buy()
        else:
            if self.data.close[0] < self.sma[0]:
                self.sell()

# Run Bond strategy
engine_bond = BacktestEngine(cash=100000, commission=0.0005)
engine_bond.add_data(IBKRDataFeed(dataname=data_dict['Bonds'].copy()), name='TLT')
engine_bond.add_strategy(BondVolatilityStrategy)
result_bond = engine_bond.run_backtest()

print("=== Bond Volatility Strategy Results ===")
print(f"Final Value: ${result_bond['final_value']:,.0f}")
print(f"Total Return: {result_bond['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {result_bond['sharpe_ratio']:.3f}")

## 6. Strategy 4: Equity Trend Following

In [ ]:
class EquityTrendStrategy(bt.Strategy):
    """
    Long-term trend following on equities.
    - 200-day SMA filter
    """
    params = (
        ('sma_period', 200),
    )
    
    def __init__(self):
        self.sma = bt.indicators.SimpleMovingAverage(
            self.data.close, period=self.params.sma_period
        )
        
    def next(self):
        if len(self) < self.params.sma_period:
            return
            
        if not self.position:
            if self.data.close[0] > self.sma[0]:
                self.buy()
        else:
            if self.data.close[0] < self.sma[0]:
                self.sell()

# Run Equity strategy
engine_equity = BacktestEngine(cash=100000, commission=0.001)
engine_equity.add_data(IBKRDataFeed(dataname=data_dict['Equities'].copy()), name='IVV')
engine_equity.add_strategy(EquityTrendStrategy)
result_equity = engine_equity.run_backtest()

print("=== Equity Trend Strategy Results ===")
print(f"Final Value: ${result_equity['final_value']:,.0f}")
print(f"Total Return: {result_equity['total_return']*100:.2f}%")
print(f"Sharpe Ratio: {result_equity['sharpe_ratio']:.3f}")

## 7. Multi-Asset Portfolio

In [ ]:
# Note: Backtrader handles multi-asset differently
# We can either:
# 1. Use cerebro.adddata() for multiple datas with one strategy
# 2. Run separate backtests and combine results

# For simplicity, let's compare all results
print("=== All Strategy Results Summary ===")

results = {
    'FX Carry': result_fx,
    'Crypto MR': result_crypto,
    'Bond Vol': result_bond,
    'Equity Trend': result_equity
}

for name, res in results.items():
    print(f"\n{name}:")
    print(f"  Return: {res['total_return']*100:.2f}%")
    print(f"  Sharpe: {res['sharpe_ratio']:.3f}")
    print(f"  Max DD: {res['max_drawdown']*100:.2f}%")

## 8. Walk-Forward Optimization

In [ ]:
# Simple walk-forward test for equity trend
# Test different SMA periods

sma_periods = [100, 150, 200, 250]
walk_forward_results = []

for sma in sma_periods:
    engine = BacktestEngine(cash=100000, commission=0.001)
    engine.add_data(IBKRDataFeed(dataname=data_dict['Equities'].copy()), name='IVV')
    
    # Create strategy with this SMA period
    class TempStrategy(bt.Strategy):
        params = (('sma_period', sma),)
        def __init__(self):
            self.sma = bt.indicators.SimpleMovingAverage(
                self.data.close, period=self.params.sma_period
            )
        def next(self):
            if len(self) < self.params.sma_period:
                return
            if not self.position:
                if self.data.close[0] > self.sma[0]:
                    self.buy()
            else:
                if self.data.close[0] < self.sma[0]:
                    self.sell()
    
    engine.add_strategy(TempStrategy)
    result = engine.run_backtest()
    walk_forward_results.append({
        'sma_period': sma,
        'return': result['total_return']*100,
        'sharpe': result['sharpe_ratio'],
        'max_dd': result['max_drawdown']*100
    })

wf_df = pd.DataFrame(walk_forward_results)
print("=== Walk-Forward Optimization ===")
print(wf_df.to_string(index=False))
print(f"\nBest SMA period: {wf_df.loc[wf_df['sharpe'].idxmax(), 'sma_period']}")

## 9. Regime Analysis

In [ ]:
# Analyze strategy by market regime
# Define regimes by equity performance

equity_data = data_dict['Equities'].copy()
equity_data['returns'] = equity_data['close'].pct_change()
equity_data['momentum_60'] = equity_data['returns'].rolling(60).sum()

# Define regimes
def categorize_regime(momentum):
    if momentum > 0.10:
        return 'Bull'
    elif momentum < -0.10:
        return 'Bear'
    else:
        return 'Neutral'

equity_data['regime'] = equity_data['momentum_60'].apply(categorize_regime)

# Get returns by regime for each strategy
print("=== Returns by Market Regime ===")
regime_returns = {}
for name, res in results.items():
    # This would require tracking trades by regime
    # For now, just show overall
    regime_returns[name] = res['total_return'] * 100

regime_df = pd.DataFrame([regime_returns]).T
regime_df.columns = ['Total Return (%)']
print(regime_df)

## 10. Transaction Cost Sensitivity

In [ ]:
# Test different commission levels
commission_levels = [0, 0.0005, 0.001, 0.002, 0.003]
cost_results = []

for comm in commission_levels:
    engine = BacktestEngine(cash=100000, commission=comm)
    engine.add_data(IBKRDataFeed(dataname=data_dict['Equities'].copy()), name='IVV')
    engine.add_strategy(EquityTrendStrategy)
    result = engine.run_backtest()
    cost_results.append({
        'commission': f"{comm*100:.2f}%",
        'return': f"{result['total_return']*100:.2f}%",
        'sharpe': f"{result['sharpe_ratio']:.3f}"
    })

cost_df = pd.DataFrame(cost_results)
print("=== Cost Sensitivity Analysis ===")
print(cost_df.to_string(index=False))
print("\nNote: Higher costs significantly impact profitability")

## Summary

In this advanced tutorial, you learned:

1. **Multi-Asset Data**: How to load FX, Crypto, Bonds, and Equities
2. **Strategy Types**: Implemented momentum, mean reversion, and trend strategies
3. **Backtrader**: How to use BacktestEngine with custom strategies
4. **Walk-Forward Optimization**: How to test parameter stability
5. **Regime Analysis**: Understanding strategy performance in different markets
6. **Cost Sensitivity**: Impact of transaction costs on profitability

**Key Takeaways:**
- Different assets require different strategy types
- Walk-forward validation helps avoid overfitting
- Transaction costs have significant impact on returns
- Diversification across strategies improves risk-adjusted returns

**Next Steps:**
- Implement more sophisticated portfolio construction
- Add machine learning for signal generation
- Paper trade and validate with live data
